# LeetCode 3996 — Even Number of Knight Moves bằng Python

Link đề: https://leetcode.com/problems/even-number-of-knight-moves/ · Độ khó: **Easy** · Chủ đề: Math, Parity, BFS, Chess

Skeleton LeetCode:

```python
class Solution:
    def canReach(self, start: List[int], target: List[int]) -> bool:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình. Slug URL được suy ra từ tên bài.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trên bàn cờ `8 x 8`, con **mã** đi từ ô `start` tới ô `target` bằng **số bước chẵn** được không? (cho phép 0 bước nếu hai ô trùng nhau).
2. Biết nước đi của quân mã (chữ L: `(±1, ±2)` và `(±2, ±1)`) và cách mô phỏng bằng **BFS trên trạng thái `(ô, tính chẵn lẻ số bước)`**.
3. Nhận ra **bước đột phá về tính chẵn lẻ**: mỗi nước đi của mã **đổi màu ô** (`x + y` đổi tính chẵn lẻ), nên số bước chẵn khi và chỉ khi hai ô **cùng màu**.
4. Viết 4 cách (BFS theo trạng thái chẵn/lẻ, tính chẵn lẻ `x + y`, XOR bit, bảng tra) và đối chiếu bằng test **vét cạn mọi cặp ô** (64 x 64).
5. Tránh lỗi: dùng khoảng cách Manhattan, chỉ so `x` hoặc chỉ so `y`, quên trường hợp `start == target`, nhầm "đi được" với "đi được bằng số bước chẵn".

## 2. Đề bài & ví dụ

Cho hai mảng `start = [x, y]` và `target = [x, y]` là tọa độ hai ô trên bàn cờ vua `8 x 8` (`0 <= x, y <= 7`). Một **nước đi của quân mã** là đi **2 ô theo một hướng ngang/dọc rồi 1 ô theo hướng vuông góc** (hình chữ L), miễn là còn nằm trong bàn cờ. Hãy trả về `True` nếu quân mã có thể đi từ `start` đến `target` bằng **một số chẵn các nước đi** (kể cả `0` nước), ngược lại `False`.

| Ví dụ | `start`, `target` | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,1]`, `[2,2]` | `True` | Một cách: `(1,1)->(3,2)->(2,4)->(4,3)->(2,2)`, tổng 4 nước (chẵn) |
| 2 | `[4,5]`, `[6,6]` | `False` | Không thể tới `[6,6]` từ `[4,5]` bằng số nước chẵn |

Constraints: `len(start) == len(target) == 2`; `0 <= start[i], target[i] <= 7`.

Đề này nằm giữa "đường đi" và "tính chẵn lẻ": ta sẽ thấy không cần tìm đường cụ thể.

In [1]:
MOVES = [(1, 2), (2, 1), (-1, 2), (-2, 1), (1, -2), (2, -1), (-1, -2), (-2, -1)]
x, y = 1, 1
print("các ô mã đi tới được từ (1,1) trong 1 nước:",
      [(x + dx, y + dy) for dx, dy in MOVES if 0 <= x + dx < 8 and 0 <= y + dy < 8])
path = [(1, 1), (3, 2), (2, 4), (4, 3), (2, 2)]
print("đường đi của ví dụ 1 có", len(path) - 1, "nước")
assert all(abs(a[0] - b[0]) * abs(a[1] - b[1]) == 2 for a, b in zip(path, path[1:]))

các ô mã đi tới được từ (1,1) trong 1 nước: [(2, 3), (3, 2), (0, 3), (3, 0)]
đường đi của ví dụ 1 có 4 nước


## 3. Kiến thức nền

### 3.1. Nước đi của quân mã

Mã đi tới `(x + dx, y + dy)` với `(dx, dy)` thuộc 8 cặp: `(±1, ±2)`, `(±2, ±1)`. Một cách kiểm tra nhanh một bước là hợp lệ: `abs(dx) * abs(dy) == 2` (chỉ `1*2` hoặc `2*1`). Phải nằm trong `0 <= x, y < 8`.

In [2]:
MOVES = [(1, 2), (2, 1), (-1, 2), (-2, 1), (1, -2), (2, -1), (-1, -2), (-2, -1)]
print(len(MOVES), all(abs(dx) * abs(dy) == 2 for dx, dy in MOVES))

8 True


### 3.2. Màu ô cờ và tính chẵn lẻ `(x + y) % 2`

Trên bàn cờ, các ô được tô xen kẽ hai màu. Màu của ô `(x, y)` xác định bởi `(x + y) % 2` (`0` hoặc `1`). Hai ô kề cạnh luôn khác màu. Phép `% 2` cho phần dư khi chia 2: `7 % 2 = 1`, `8 % 2 = 0`.

In [3]:
for x in range(4):
    print(" ".join(str((x + y) % 2) for y in range(8)))

0 1 0 1 0 1 0 1
1 0 1 0 1 0 1 0
0 1 0 1 0 1 0 1
1 0 1 0 1 0 1 0


### 3.3. Mỗi nước đi của mã đổi màu ô

Nước đi có `dx + dy` là `±1 ± 2` hoặc `±2 ± 1`, tức `dx + dy ∈ {±1, ±3}`: luôn là **số lẻ**. Do đó `(x + y) % 2` luôn **đổi** sau mỗi nước đi.

In [4]:
print(sorted({dx + dy for dx, dy in MOVES}), all((dx + dy) % 2 == 1 for dx, dy in MOVES))

[-3, -1, 1, 3] True


### 3.4. BFS (duyệt theo tầng) trên đồ thị trạng thái

BFS dùng hàng đợi `collections.deque`: lấy trạng thái ở đầu hàng, sinh các trạng thái kề, đánh dấu đã thăm (`set`), đẩy vào cuối hàng. Dùng cho "số bước ít nhất" hoặc "có tới được không". Ở bài này **trạng thái** là `(x, y, p)` với `p` là tính chẵn lẻ của số bước đã đi.

In [5]:
from collections import deque
q = deque([(1, 1, 0)])
seen = {(1, 1, 0)}
print(q.popleft(), len(q), (1, 1, 0) in seen)

(1, 1, 0) 0 True


### 3.5. XOR và bit thấp nhất

`a ^ b` là XOR bit. Bit thấp nhất của `a ^ b` bằng `1` khi `a` và `b` có tính chẵn lẻ **khác nhau**. Vậy `((x ^ y ^ tx ^ ty) & 1) == 0` nghĩa là `(x + y) % 2 == (tx + ty) % 2`. Đây chỉ là cách viết gọn bằng bit.

In [6]:
a, b = 3, 8
print(a ^ b, (a ^ b) & 1, (a + b) % 2, (3 + 5) % 2, (3 ^ 5) & 1)

11 1 1 0 0


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Cách nghĩ thẳng thắn: coi mỗi ô cờ là một đỉnh, mỗi nước đi của mã là một cạnh; ta muốn biết **có đường đi độ dài chẵn** từ `start` đến `target` không. Muốn kiểm tra "chẵn/lẻ" ta nhớ thêm **tính chẵn lẻ của số bước** vào trạng thái.

Tính tay ví dụ 2: `start = (4, 5)` màu `(4 + 5) % 2 = 1`; `target = (6, 6)` màu `(6 + 6) % 2 = 0`. Mỗi nước đi đổi màu, nên sau `k` nước màu là `1 + k (mod 2)`. Để tới màu `0` cần `k` **lẻ**. Vậy không thể bằng số nước chẵn -> `False`.

Tính tay ví dụ 1: `(1, 1)` màu `0`, `(2, 2)` màu `0`: cùng màu -> số nước chẵn -> `True`.

| `start` | màu | `target` | màu | cùng màu? | kết quả |
|---|---|---|---|---|---|
| `(1,1)` | 0 | `(2,2)` | 0 | có | `True` |
| `(4,5)` | 1 | `(6,6)` | 0 | không | `False` |
| `(0,0)` | 0 | `(0,0)` | 0 | có (0 nước) | `True` |
| `(0,0)` | 0 | `(0,1)` | 1 | không | `False` |

In [7]:
def color(p):
    return (p[0] + p[1]) % 2

for s, t in [((1, 1), (2, 2)), ((4, 5), (6, 6)), ((0, 0), (0, 0)), ((0, 0), (0, 1))]:
    print(s, color(s), t, color(t), "cùng màu?", color(s) == color(t))

(1, 1) 0 (2, 2) 0 cùng màu? True
(4, 5) 1 (6, 6) 0 cùng màu? False
(0, 0) 0 (0, 0) 0 cùng màu? True
(0, 0) 0 (0, 1) 1 cùng màu? False


## 5. Cách 1 — Brute Force / baseline: BFS theo trạng thái `(x, y, parity)`

Bắt đầu từ `(sx, sy, 0)` (đã đi 0 bước, chẵn). Mỗi nước đi chuyển sang ô kề và **đảo `parity`**. Nếu chạm trạng thái `(tx, ty, 0)` thì trả `True`. Hết trạng thái mà chưa thấy thì `False`.

Số trạng thái tối đa `8 * 8 * 2 = 128`, nhỏ; nhưng ta **đang mô phỏng** điều có thể suy ra bằng lập luận chẵn lẻ (nút thắt về mặt tư duy).

In [8]:
from collections import deque

class SolutionBFS:
    def canReach(self, start, target):
        sx, sy = start
        tx, ty = target
        seen = {(sx, sy, 0)}
        q = deque([(sx, sy, 0)])
        while q:
            x, y, p = q.popleft()
            if (x, y, p) == (tx, ty, 0):
                return True
            for dx, dy in MOVES:
                nx, ny = x + dx, y + dy
                if 0 <= nx < 8 and 0 <= ny < 8 and (nx, ny, 1 - p) not in seen:
                    seen.add((nx, ny, 1 - p))
                    q.append((nx, ny, 1 - p))
        return False

print(SolutionBFS().canReach([1, 1], [2, 2]), SolutionBFS().canReach([4, 5], [6, 6]))

True False


**Độ phức tạp:** Time `O(8 * 8 * 2 * 8) = O(1)` (hằng số lớn vì bàn cờ cố định), Space `O(128)` trạng thái.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá (bất biến chẵn lẻ):** mỗi nước đi của mã **luôn đổi màu ô**. Do đó sau `k` nước, màu ô hiện tại là `(màu_ban_đầu + k) mod 2`. Nên:

- nếu `k` **chẵn** -> **cùng màu** với ô xuất phát;
- nếu `k` **lẻ** -> **khác màu**.

Như vậy điều kiện **cần**: `start` và `target` cùng màu. Điều kiện **đủ** (cho bàn cờ `8 x 8`): mọi ô cùng màu đều tới được bằng số bước chẵn (kể cả `0` bước khi trùng nhau). Ta kiểm chứng điều này bằng BFS trên toàn bộ 64 x 64 cặp ô ở mục 12.

**Quy tắc:** `True` khi và chỉ khi `(sx + sy) % 2 == (tx + ty) % 2`.

**Invariant:** `(x + y) % 2` của vị trí con mã thay đổi đúng một lần với mỗi nước đi; hai nước liên tiếp quay về cùng màu.

## 7. Các cách tiếp theo

### Cách 2 — So sánh `(x + y) % 2` của hai ô

Đúng theo bước đột phá, `O(1)`.

In [9]:
class SolutionParity:
    def canReach(self, start, target):
        return (start[0] + start[1]) % 2 == (target[0] + target[1]) % 2

print(SolutionParity().canReach([1, 1], [2, 2]), SolutionParity().canReach([4, 5], [6, 6]))

True False


**Độ phức tạp:** Time `O(1)`, Space `O(1)`.

### Cách 3 — XOR bit thấp nhất

`(sx ^ sy ^ tx ^ ty) & 1 == 0` nghĩa là tổng tính chẵn lẻ bốn tọa độ bằng chẵn, tương đương cùng màu.

In [10]:
class SolutionXor:
    def canReach(self, start, target):
        return ((start[0] ^ start[1] ^ target[0] ^ target[1]) & 1) == 0

print(SolutionXor().canReach([1, 1], [2, 2]), SolutionXor().canReach([4, 5], [6, 6]))

True False


**Độ phức tạp:** Time `O(1)`, Space `O(1)`.

### Cách 4 — Bảng tra: tính trước tập ô theo màu

Dựng sẵn `COLOR[x][y]` (bảng `8 x 8` các giá trị `0/1`), rồi trả `COLOR[sx][sy] == COLOR[tx][ty]`. Tách "luật" (cách tô màu) khỏi "truy vấn", hữu ích nếu luật phức tạp hơn.

In [11]:
COLOR = [[(x + y) % 2 for y in range(8)] for x in range(8)]

class SolutionTable:
    def canReach(self, start, target):
        return COLOR[start[0]][start[1]] == COLOR[target[0]][target[1]]

print(SolutionTable().canReach([1, 1], [2, 2]), SolutionTable().canReach([4, 5], [6, 6]))

True False


**Độ phức tạp:** Time `O(1)`, Space `O(64)` bảng cố định.

## 8. TRACE từng bước

Trace Cách 1 (BFS) với `start = (1, 1)`, `target = (2, 2)` ở vài tầng đầu:

| tầng (số bước) | tính chẵn lẻ `p` | một vài ô tới được |
|---|---|---|
| 0 | 0 | `(1,1)` |
| 1 | 1 | `(2,3)`, `(3,2)`, `(0,3)`, `(3,0)` |
| 2 | 0 | `(1,1)`, `(2,2)` (qua `(3,0)`), `(0,5)`, ... |

Ví dụ của đề dùng 4 nước nhưng thực tế chỉ cần **2** nước chẵn: `(1,1) -> (3,0) -> (2,2)`. Code dưới in số nước chẵn ít nhất.

In [12]:
def trace_bfs(start, target):
    sx, sy = start
    dist = {(sx, sy, 0): 0}
    q = deque([(sx, sy, 0)])
    while q:
        x, y, p = q.popleft()
        for dx, dy in MOVES:
            nx, ny = x + dx, y + dy
            if 0 <= nx < 8 and 0 <= ny < 8 and (nx, ny, 1 - p) not in dist:
                dist[(nx, ny, 1 - p)] = dist[(x, y, p)] + 1
                q.append((nx, ny, 1 - p))
    d_even = dist.get((target[0], target[1], 0))
    d_odd = dist.get((target[0], target[1], 1))
    print(f"{start} -> {target}: số bước chẵn ít nhất = {d_even}, số bước lẻ ít nhất = {d_odd}")
    return d_even

assert trace_bfs((1, 1), (2, 2)) == 2          # ví dụ 1 dùng 4 nước, nhưng chỉ cần 2: (1,1)->(3,0)->(2,2)
assert trace_bfs((4, 5), (6, 6)) is None       # không có đường chẵn
assert trace_bfs((0, 0), (0, 0)) == 0          # 0 nước

(1, 1) -> (2, 2): số bước chẵn ít nhất = 2, số bước lẻ ít nhất = None
(4, 5) -> (6, 6): số bước chẵn ít nhất = None, số bước lẻ ít nhất = 1
(0, 0) -> (0, 0): số bước chẵn ít nhất = 0, số bước lẻ ít nhất = None


In [13]:
def trace_parity(start, target):
    cs = (start[0] + start[1]) % 2
    ct = (target[0] + target[1]) % 2
    print(f"màu({start}) = ({start[0]} + {start[1]}) % 2 = {cs}")
    print(f"màu({target}) = ({target[0]} + {target[1]}) % 2 = {ct}")
    print("  cùng màu -> True" if cs == ct else "  khác màu -> False")
    return cs == ct

assert trace_parity((1, 1), (2, 2)) is True
assert trace_parity((4, 5), (6, 6)) is False

màu((1, 1)) = (1 + 1) % 2 = 0
màu((2, 2)) = (2 + 2) % 2 = 0
  cùng màu -> True
màu((4, 5)) = (4 + 5) % 2 = 1
màu((6, 6)) = (6 + 6) % 2 = 0
  khác màu -> False


## 9. Code nộp LeetCode

Dùng Cách 2 (`O(1)`).

In [14]:
from typing import List, Optional


class Solution:
    def canReach(self, start: List[int], target: List[int]) -> bool:
        return (start[0] + start[1]) % 2 == (target[0] + target[1]) % 2


print(Solution().canReach([1, 1], [2, 2]), Solution().canReach([4, 5], [6, 6]))

True False


## 10. Vì sao thuật toán đúng

1. **Điều kiện cần.** Mỗi nước đi có `dx + dy` lẻ (xem mục 3.3), nên làm đổi `(x + y) % 2`. Sau `k` nước, `(x + y) % 2` đã đổi `k` lần, tức bằng giá trị ban đầu cộng `k` (mod 2). Số nước **chẵn** => cùng màu với điểm xuất phát. Do đó nếu `start` và `target` khác màu thì **không thể** có đường đi chẵn: kết quả `False`.
2. **Điều kiện đủ (trên bàn `8 x 8`).** Nếu hai ô cùng màu, tồn tại đường đi chẵn. Với `start == target` ta dùng 0 nước. Với `start != target` cùng màu, bàn cờ `8 x 8` đủ lớn nên con mã đi được mọi ô cùng màu bằng số bước chẵn (đồ thị hai-bước của mã trên mỗi màu liên thông). Thay vì chứng minh bằng lời, ta **kiểm chứng đầy đủ bằng BFS** cho cả `64 * 64 = 4096` cặp (mục 12): không có cặp nào trái với công thức.
3. **Tương đương các cách.** Cách 3 (XOR) chỉ là cách viết khác của `(sx + sy) % 2 == (tx + ty) % 2`; Cách 4 tra bảng cùng giá trị.
4. **Lưu ý quan trọng:** phần "điều kiện đủ" phụ thuộc kích thước bàn cờ (`8 x 8`). Trên bàn cờ nhỏ (ví dụ `3 x 3`) con mã không tới được ô trung tâm và công thức sẽ sai, nên bài này cần ràng buộc kích thước cố định.

## 11. Phân tích Time / Space Complexity

- Cách 1 (BFS): tối đa `128` trạng thái, mỗi trạng thái 8 nước đi: Time `O(1)` (hằng số), Space `O(1)`.
- Cách 2, 3: Time `O(1)`, Space `O(1)`.
- Cách 4: Time `O(1)`, Space `O(64)` bảng.

Với bàn cờ cố định, mọi cách là `O(1)`. Nếu bàn cờ `N x N`, BFS là `O(N^2)` còn công thức chẵn lẻ vẫn `O(1)` (khi `N` đủ lớn để thỏa điều kiện đủ).

## 12. Bộ test

### Cách kiểm thử trong notebook này

Chúng ta dùng ba lớp kiểm thử, từ dễ đến khó:

1. **Test cố định**: ví dụ của đề + các trường hợp biên tự nghĩ ra. Mỗi test là một `assert hàm(input) == kết_quả_mong_đợi`. Nếu sai, Python ném `AssertionError` và dừng ngay, kèm thông tin ta in ra để dễ lần lại.
2. **Test ngẫu nhiên có `random.seed`**: gọi `random.seed(số)` trước vòng lặp để **mỗi lần chạy lại sinh đúng cùng dữ liệu** (tái lập được). Input luôn được sinh **trong đúng constraints** của đề.
3. **Đối chiếu nhiều cách (oracle)**: một cách rất đơn giản, hiển nhiên đúng (brute force) đóng vai "trọng tài"; mọi cách nhanh hơn phải cho cùng kết quả với trọng tài trên mọi input ngẫu nhiên.

Vì miền đầu vào chỉ có `64 * 64 = 4096` cặp ô, ta còn có thể **vét cạn toàn bộ** để chứng minh công thức chẵn lẻ đúng.

In [15]:
import random

solutions = [SolutionBFS(), SolutionParity(), SolutionXor(), SolutionTable(), Solution()]

fixed = [
    ([1, 1], [2, 2], True), ([4, 5], [6, 6], False), ([0, 0], [0, 0], True), ([0, 0], [0, 1], False),
    ([0, 0], [7, 7], True), ([0, 0], [7, 6], False), ([3, 3], [3, 3], True), ([7, 7], [0, 0], True),
    ([0, 7], [7, 0], True), ([2, 3], [2, 4], False), ([2, 3], [4, 3], True),
]
for s, t, expected in fixed:
    for sol in solutions:
        assert sol.canReach(list(s), list(t)) == expected, (type(sol).__name__, s, t)
print("test cố định OK:", len(fixed), "ca")

# vét cạn toàn miền: 8*8*8*8 = 4096 cặp (BFS là trọng tài)
cells = [(x, y) for x in range(8) for y in range(8)]
true_count = 0
for s in cells:
    for t in cells:
        expected = SolutionBFS().canReach(list(s), list(t))
        true_count += expected
        for sol in solutions[1:]:
            assert sol.canReach(list(s), list(t)) == expected, (s, t)
print("vét cạn 4096 cặp OK; số cặp True =", true_count)
assert true_count == 2 * 32 * 32           # mỗi màu có 32 ô: 32*32 cặp cùng màu, hai màu

random.seed(3996)
for _ in range(500):
    s = [random.randint(0, 7), random.randint(0, 7)]
    t = [random.randint(0, 7), random.randint(0, 7)]
    assert len({sol.canReach(list(s), list(t)) for sol in solutions}) == 1
print("test ngẫu nhiên OK")

test cố định OK: 11 ca


vét cạn 4096 cặp OK; số cặp True = 2048


test ngẫu nhiên OK


### Test thuộc tính (property test)

(1) **Đối xứng:** `canReach(a, b) == canReach(b, a)`. (2) **Phản xạ:** `canReach(a, a)` luôn `True`. (3) **Bắc cầu:** nếu `a ~ b` và `b ~ c` thì `a ~ c` (cùng màu là quan hệ tương đương). (4) Không sửa mảng đầu vào.

In [16]:
random.seed(39960)
for _ in range(500):
    a, b, c = ([random.randint(0, 7), random.randint(0, 7)] for _ in range(3))
    S = Solution()
    snap = (list(a), list(b))
    assert S.canReach(a, b) == S.canReach(b, a)
    assert S.canReach(a, a) is True
    if S.canReach(a, b) and S.canReach(b, c):
        assert S.canReach(a, c)
    assert (a, b) == snap
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng khoảng cách Manhattan hay Chebyshev

Mã không đi theo đường thẳng nên **số nước đi không bằng** `d // 2` hay `d / 3` với `d = |dx| + |dy|`. Ví dụ `(1,1) -> (2,2)` có `d = 2`; công thức «số nước = d // 2 = 1» cho số **lẻ**, nhưng thực tế cần `2` nước (chẵn) nên đáp án là `True`.

In [17]:
def wrong_distance_even(start, target):
    d = abs(start[0] - target[0]) + abs(start[1] - target[1])
    return d // 2 % 2 == 0                                      # SAI: khoảng cách != số nước đi

print(wrong_distance_even([1, 1], [2, 2]), "!=", Solution().canReach([1, 1], [2, 2]))
assert wrong_distance_even([1, 1], [2, 2]) is False and Solution().canReach([1, 1], [2, 2]) is True

False != True


### Lỗi 2: chỉ so sánh tọa độ `x` (hoặc chỉ `y`)

Màu ô phụ thuộc vào **tổng** `x + y`, không phải riêng `x` hay `y`.

In [18]:
def wrong_x_only(start, target):
    return start[0] % 2 == target[0] % 2                        # SAI: bỏ qua y

print(wrong_x_only([0, 0], [0, 1]), "!=", Solution().canReach([0, 0], [0, 1]))
assert wrong_x_only([0, 0], [0, 1]) is True and Solution().canReach([0, 0], [0, 1]) is False

True != False


### Lỗi 3: nhầm "tới được" với "tới được bằng số bước chẵn"

Trên bàn `8 x 8`, mã tới được **mọi** ô (nhiều ô cần số nước lẻ). Trả `True` luôn cho "tới được" bỏ qua yêu cầu chẵn.

In [19]:
def wrong_always_reachable(start, target):
    return True                                                 # SAI: bỏ qua điều kiện "chẵn"

print(wrong_always_reachable([4, 5], [6, 6]), "!=", Solution().canReach([4, 5], [6, 6]))
assert wrong_always_reachable([4, 5], [6, 6]) is True and Solution().canReach([4, 5], [6, 6]) is False

True != False


### Lỗi 4: đếm `start == target` là `False` (quên 0 nước là số chẵn)

`0` là số chẵn nên đứng yên tại chỗ là một đường đi chẵn.

In [20]:
def wrong_same_cell(start, target):
    if start == target:
        return False                                            # SAI: 0 nước là chẵn
    return (start[0] + start[1]) % 2 == (target[0] + target[1]) % 2

print(wrong_same_cell([3, 3], [3, 3]), "!=", Solution().canReach([3, 3], [3, 3]))
assert wrong_same_cell([3, 3], [3, 3]) is False and Solution().canReach([3, 3], [3, 3]) is True

False != True


### Lỗi 5: đảo ngược kết luận chẵn/lẻ

Cùng màu <=> chẵn. Viết `!=` (khác màu) cho kết quả ngược với đáp án.

In [21]:
def wrong_inverted(start, target):
    return (start[0] + start[1]) % 2 != (target[0] + target[1]) % 2    # SAI: đảo ngược

print(wrong_inverted([1, 1], [2, 2]), "!=", Solution().canReach([1, 1], [2, 2]))
assert wrong_inverted([1, 1], [2, 2]) is False and Solution().canReach([1, 1], [2, 2]) is True

False != True


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài "đường đi có số bước **chẵn/lẻ**" -> nghĩ ngay tới **bất biến chẵn lẻ** (tô màu xen kẽ). Mã đổi màu mỗi nước; xe/hậu/tượng có tính chất khác.
- `(x + y) % 2` là "màu ô cờ"; cùng màu <=> hiệu `x + y` chẵn.
- Khi nghi ngờ, **BFS vét cạn miền nhỏ** để kiểm chứng giả thuyết (ở đây 4096 cặp): kỹ thuật "đoán rồi chứng minh bằng máy" rất đáng giá.
- Nhớ điều kiện đủ phụ thuộc kích thước bàn cờ (bàn nhỏ có ô không tới được).
- Đồ thị có thêm thông tin phụ (chẵn/lẻ số bước) -> BFS trên **trạng thái mở rộng** `(đỉnh, parity)`.
- Ca biên: hai ô trùng nhau, hai ô ở hai góc đối diện, hai ô kề nhau.

## 15. Biến thể của bài

1. **Minimum Knight Moves** (1197, Premium): số nước đi ít nhất của mã trên bàn vô hạn.
2. **Knight Probability in Chessboard** (688), **Check Knight Tour Configuration** (2596).
3. **Number of Ways to Reach a Position After Exactly k Steps** (2400): đếm đường đi theo số bước.
4. Biến thể: số nước **lẻ** (trả về `not canReach`) trên bàn `8 x 8`.
5. Biến thể: bàn cờ `N x N` nhỏ (`N = 3, 4`): cần BFS thật vì công thức chẵn lẻ không đủ.
6. Biến thể: số nước ít nhất để tới (không chỉ chẵn/lẻ): BFS từ `start` trên bàn `8 x 8`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS `(x, y, parity)` | mô phỏng nước đi | `O(1)` (128 trạng thái) | `O(1)` | Trọng tài; bàn cờ bất kỳ |
| 2. `(x + y) % 2` | cùng màu <=> chẵn | `O(1)` | `O(1)` | **Nộp bài** |
| 3. XOR bit thấp | `(a ^ b ^ c ^ d) & 1 == 0` | `O(1)` | `O(1)` | Viết gọn bằng bit |
| 4. Bảng màu | tra `COLOR[x][y]` | `O(1)` | `O(64)` | Khi luật tô màu phức tạp |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().canReach([0, 0], [1, 2])` và `Solution().canReach([5, 5], [2, 1])` ra gì?

**Bài 2 (tính tay):** `(0, 0)` tới `(2, 1)` bằng mấy nước (ít nhất)? Số nước lẻ hay chẵn? Giải thích bằng màu ô.

**Bài 3 (điền TODO):** hoàn thành `can_reach_odd(start, target)`: tới được bằng số nước **lẻ** (không tính 0 nước).

**Bài 4 (tự viết lại):** viết `min_moves(start, target)` bằng BFS trên bàn `8 x 8` (số nước ít nhất).

**Bài 5 (mini challenge):** đếm có bao nhiêu ô `(x, y)` mà mã từ `(0, 0)` tới được bằng số nước chẵn; so với số ô cùng màu.

In [22]:
# Bài 1: đoán rồi chạy
print(Solution().canReach([0, 0], [1, 2]), Solution().canReach([5, 5], [2, 1]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def can_reach_odd_todo(start, target):
    # TODO: số nước lẻ <=> khác màu (nhớ: trùng nhau cần 0 nước, KHÔNG phải lẻ)
    return False

False False


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: tính màu của hai ô mỗi cặp. Bài 2: `(2, 1)` chỉ cách `(0, 0)` đúng một nước đi chữ L.

**Mức 2 (vừa):** Bài 3: khác màu thì chắc chắn có đường lẻ (đối xứng với chẵn); cùng màu và khác ô thì cũng có đường lẻ? Hãy kiểm tra bằng BFS (trạng thái parity 1). Bài 4: BFS với `dist`. Bài 5: duyệt 64 ô, gọi `canReach`.

**Mức 3 (gần đáp án):** Bài 1: `[0,0] -> [1,2]` khác màu (`0` và `1`) -> `False`; `[5,5] -> [2,1]` màu `0` và `1` -> `False`. Bài 2: `1` nước, lẻ; khác màu. Bài 3: chỉ cần so màu khác nhau. Bài 5: `32`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
assert Solution().canReach([0, 0], [1, 2]) is False and Solution().canReach([5, 5], [2, 1]) is False
assert color((0, 0)) == 0 and color((2, 1)) == 1

# Bài 3
def can_reach_odd(start, target):
    return (start[0] + start[1]) % 2 != (target[0] + target[1]) % 2

# kiểm chứng bằng BFS: tính số bước lẻ ít nhất (parity 1) cho mọi ô
def dist_parity(start):
    sx, sy = start
    dist = {(sx, sy, 0): 0}
    q = deque([(sx, sy, 0)])
    while q:
        x, y, p = q.popleft()
        for dx, dy in MOVES:
            nx, ny = x + dx, y + dy
            if 0 <= nx < 8 and 0 <= ny < 8 and (nx, ny, 1 - p) not in dist:
                dist[(nx, ny, 1 - p)] = dist[(x, y, p)] + 1
                q.append((nx, ny, 1 - p))
    return dist

for s in [(0, 0), (3, 4), (7, 7)]:
    d = dist_parity(s)
    for t in cells:
        assert can_reach_odd(list(s), list(t)) == ((t[0], t[1], 1) in d), (s, t)

# Bài 4
def min_moves(start, target):
    d = dist_parity(start)
    options = [d[k] for k in ((target[0], target[1], 0), (target[0], target[1], 1)) if k in d]
    return min(options)

assert min_moves((0, 0), (0, 0)) == 0 and min_moves((0, 0), (2, 1)) == 1 and min_moves((0, 0), (7, 7)) == 6

# Bài 5
reachable_even = sum(1 for t in cells if Solution().canReach([0, 0], list(t)))
assert reachable_even == 32 == sum(1 for t in cells if color(t) == color((0, 0)))
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
class Solution:
    def canReach(self, start: List[int], target: List[int]) -> bool:
        # mã luôn đổi màu ô sau mỗi nước đi: số nước chẵn <=> cùng màu
        return (start[0] + start[1]) % 2 == (target[0] + target[1]) % 2
```

Khung chung "đường đi chẵn/lẻ": tìm **bất biến chẵn lẻ**; nếu nghi ngờ thì BFS `(đỉnh, parity)`.

### Checklist

- [ ] Đã đọc kỹ constraints (độ dài, miền giá trị) và biết thuật toán chạy được trong giới hạn đó.
- [ ] Đã chạy tay ít nhất **một ví dụ của đề** và một **trường hợp biên** (nhỏ nhất, lớn nhất, giá trị đặc biệt).
- [ ] Đúng **tên class / tên hàm / kiểu trả về** như skeleton LeetCode (đúng `bool`, `int`, `list`...).
- [ ] Không dùng `input()`, không `print` thừa trong code nộp.
- [ ] Không làm hỏng dữ liệu đầu vào nếu đề không cho phép (sao chép trước khi sửa).
- [ ] Đã so sánh với một cách brute force trên input ngẫu nhiên nhỏ.
- [ ] Hiểu vì sao mỗi nước đi của mã đổi màu ô (`dx + dy` luôn lẻ) và vì sao `start == target` cho `True` (0 nước là chẵn).
- [ ] Đã kiểm chứng công thức bằng BFS trên **toàn bộ 4096 cặp** ô của bàn `8 x 8`.